# Day 9 — HOL 1: Implement Watermark-Based Incremental Load (Control Table)

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 2 — Watermark-Based Incremental Loading |
| **Duration** | 60 minutes |
| **Output** | A working, hand-built cursor/watermark incremental loader with its own control table |

### Why build this by hand?
The real `orders_data_ingestion_cdc` pipeline (Day 2) already does cursor-based incremental loading for you — Lakeflow Connect manages the watermark internally, you never see it. That's great in production, but it means the mechanism is a black box. This lab has you build the exact same idea yourself, against a small practice table you control end-to-end, so "cursor-based incremental loading" stops being magic.

### Learning Objectives
- Build a control table that tracks a watermark value per source table
- Implement an incremental read using `WHERE <cursor_column> > :last_watermark`
- Advance the watermark **only after** a successful downstream write
- Prove, with your own test data, that this approach — like the real pipeline — misses hard deletes

### Setup note
This lab uses a **practice schema**, not the shared `gbmart` catalog — replace `YOUR_SCHEMA` below with something unique to you (e.g. your name). You are not touching any shared GlobalMart table in this lab.

---

> ### ⚠️ Illustrative Pattern — Not GlobalMart's Real Pipeline
>
> Be honest with yourself about what this lab is and isn't. GlobalMart's real `orders`/`order_items` incremental
> load is handled entirely by the Lakeflow-Connect-managed **`orders_data_ingestion_cdc`** pipeline (built Day 2,
> traced end-to-end in ILT 2) — a managed connector that already tracks its own cursor internally. Nobody at
> GlobalMart hand-writes a control table for it, and you never will either.
>
> So why build one by hand today? Because this is the general-purpose technique you'd reach for on any source
> that **isn't** sitting behind a managed CDC connector — a vendor API with no CDC option, an internal system
> nobody's wired up to Lakeflow Connect yet, a one-off migration job. It's a real, widely-used pattern in the
> industry — just not literally what GlobalMart's own `orders`/`order_items` pipeline does today. Think of it
> like practicing long division by hand: a calculator does it for you every day, but doing it by hand once is
> what makes the calculator's answer make sense instead of feeling like magic.
>
> Everything you build below lives in your own `main.YOUR_SCHEMA` practice schema — you are never touching
> `gbmart` in this lab.

In [ ]:
# ─── Configuration — edit YOUR_SCHEMA before running anything else ────────────
YOUR_SCHEMA = "YOUR_SCHEMA"          # e.g. "virinchy_practice" — your own sandbox schema
CATALOG     = "main"                 # using main.YOUR_SCHEMA keeps this off the shared gbmart catalog entirely

SOURCE_TABLE  = f"{CATALOG}.{YOUR_SCHEMA}.practice_upstream_orders"
TARGET_TABLE  = f"{CATALOG}.{YOUR_SCHEMA}.practice_incremental_orders"
CONTROL_TABLE = f"{CATALOG}.{YOUR_SCHEMA}.practice_ingestion_control"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{YOUR_SCHEMA}")
print(f"Working in: {CATALOG}.{YOUR_SCHEMA}")

## Phase A — Simulate an Upstream Source With a Watermark Column

Stand in for "a Postgres table with an `updated_at` trigger" using a small Delta table you fully control — this lets you *cause* changes on demand in Phase C, instead of waiting for real data to change.

In [ ]:
from pyspark.sql.functions import *

spark.sql(f"DROP TABLE IF EXISTS {SOURCE_TABLE}")

# 5 starter rows, all with the same updated_at — this is the "initial load" state,
# analogous to GlobalMart's real orders table before any incremental runs happened.
initial_df = spark.createDataFrame(
    [(f"ORD-{i:03d}", 100.0 * i, "2026-06-01T09:00:00") for i in range(1, 6)],
    ["order_id", "amount", "updated_at"]
).withColumn("updated_at", to_timestamp("updated_at"))

initial_df.write.format("delta").mode("overwrite").saveAsTable(SOURCE_TABLE)
print(f"Seeded {SOURCE_TABLE} with 5 rows")
spark.table(SOURCE_TABLE).orderBy("order_id").show()

## Phase B — Build the Control Table and Run the Initial Load

The control table holds exactly one row per source table: the highest `updated_at` value successfully loaded so far. Before any run has happened, that value is a very old timestamp — meaning "everything is new."

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
        source_table          STRING,
        last_watermark_value  TIMESTAMP,
        last_run_at           TIMESTAMP
    ) USING DELTA
""")

# Seed the control row if it doesn't exist yet — 1970-01-01 means "nothing loaded yet"
existing = spark.sql(f"SELECT * FROM {CONTROL_TABLE} WHERE source_table = 'practice_upstream_orders'")
if existing.count() == 0:
    spark.sql(f"""
        INSERT INTO {CONTROL_TABLE}
        VALUES ('practice_upstream_orders', TIMESTAMP('1970-01-01 00:00:00'), NULL)
    """)
    print("Control row initialized at epoch — next read will pull everything.")
else:
    print("Control row already exists:")
    existing.show()

In [ ]:
def run_incremental_load():
    """
    The core watermark pattern, in one function:
      1. Read the current watermark from the control table.
      2. Pull only source rows newer than that watermark.
      3. If there's anything to load, MERGE it into the target (upsert on order_id).
      4. Only on success, advance the watermark to the max updated_at just loaded.
    Steps 3 and 4 happen in that order deliberately — advancing the watermark
    before confirming the write would lose data on a failed run.
    """
    from delta.tables import DeltaTable

    # Step 1 — read the watermark
    last_watermark = spark.sql(
        f"SELECT last_watermark_value FROM {CONTROL_TABLE} WHERE source_table = 'practice_upstream_orders'"
    ).collect()[0]["last_watermark_value"]
    print(f"Watermark before this run: {last_watermark}")

    # Step 2 — pull only what's newer
    incremental_df = spark.table(SOURCE_TABLE).filter(col("updated_at") > lit(last_watermark))
    new_row_count = incremental_df.count()
    print(f"Rows newer than watermark: {new_row_count}")

    if new_row_count == 0:
        print("Nothing new — skipping write and watermark update.")
        return

    incremental_df.select("order_id", "amount", "updated_at").show()

    # Step 3 — upsert into target (create it first on the very first run)
    if not spark.catalog.tableExists(TARGET_TABLE):
        incremental_df.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
        print(f"Created {TARGET_TABLE} with initial {new_row_count} row(s)")
    else:
        target = DeltaTable.forName(spark, TARGET_TABLE)
        (target.alias("tgt")
            .merge(incremental_df.alias("src"), "tgt.order_id = src.order_id")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print(f"Merged {new_row_count} row(s) into {TARGET_TABLE}")

    # Step 4 — only now, advance the watermark
    new_watermark = incremental_df.agg(spark_max("updated_at")).collect()[0][0]
    spark.sql(f"""
        UPDATE {CONTROL_TABLE}
        SET last_watermark_value = TIMESTAMP('{new_watermark}'), last_run_at = current_timestamp()
        WHERE source_table = 'practice_upstream_orders'
    """)
    print(f"Watermark advanced to: {new_watermark}")

In [ ]:
# Run 1 — the initial load. Expect all 5 rows to come through (watermark started at epoch).
run_incremental_load()

In [ ]:
print(f"{TARGET_TABLE} row count: {spark.table(TARGET_TABLE).count()}  (expect 5)")
spark.sql(f"SELECT * FROM {CONTROL_TABLE}").show(truncate=False)

## Phase C — Prove Incremental Behavior: Run Again With No Changes

Run the exact same function again, right now, with nothing changed upstream. This is the real proof that it's incremental, not just "happens to work the first time."

In [ ]:
# Run 2 — nothing changed upstream. Expect "Rows newer than watermark: 0" and no write.
run_incremental_load()

## Phase D — Simulate an Upstream Change, Then Load Incrementally

Update one existing row's `amount` (bumping its `updated_at`, exactly like GlobalMart's real `trg_orders_updated_at` trigger would) and insert one brand-new row. Then run the loader a third time and confirm **only** these 2 rows move — not all 6.

In [ ]:
# Simulate: ORD-002's amount changes, and a brand-new ORD-006 arrives.
spark.sql(f"""
    UPDATE {SOURCE_TABLE}
    SET amount = 999.0, updated_at = TIMESTAMP('2026-06-02T10:00:00')
    WHERE order_id = 'ORD-002'
""")

spark.sql(f"""
    INSERT INTO {SOURCE_TABLE} VALUES ('ORD-006', 600.0, TIMESTAMP('2026-06-02T10:05:00'))
""")

print("Simulated 1 update (ORD-002) + 1 insert (ORD-006) in the upstream source.")

In [ ]:
# Run 3 — expect "Rows newer than watermark: 2", showing exactly ORD-002 and ORD-006.
run_incremental_load()

In [ ]:
print(f"{TARGET_TABLE} row count: {spark.table(TARGET_TABLE).count()}  (expect 6 — the merge upserted ORD-002 and inserted ORD-006)")
spark.table(TARGET_TABLE).orderBy("order_id").show()

## Phase E — Prove the Blind Spot Yourself

Now delete a row from the upstream source and run the loader again. If watermark-based loading truly can't see deletes, `ORD-001` should still be sitting in your target table, undeleted, forever — with no error telling you it's stale.

In [ ]:
spark.sql(f"DELETE FROM {SOURCE_TABLE} WHERE order_id = 'ORD-001'")
print("ORD-001 deleted from the upstream source.")
print(f"Upstream row count now: {spark.table(SOURCE_TABLE).count()}  (expect 5)")

In [ ]:
# Run 4 — expect "Rows newer than watermark: 0". The delete produced no row with a
# newer updated_at, so the loader has no idea anything happened.
run_incremental_load()

In [ ]:
still_there = spark.table(TARGET_TABLE).filter("order_id = 'ORD-001'").count()
print(f"ORD-001 still present in {TARGET_TABLE}: {still_there == 1}")
print("This is the exact same blind spot GlobalMart's real orders_data_ingestion_cdc")
print("pipeline has (Day 2) — a deleted order stays in Bronze forever unless something")
print("else (a periodic full reconciliation, or switching to CDF) catches it.")

## Submission Checklist
- [ ] Control table created and seeded with an epoch watermark
- [ ] Run 1 (initial load) loaded all 5 starter rows
- [ ] Run 2 (no changes) loaded 0 rows — proves it's incremental, not a full reload
- [ ] Run 3 picked up exactly the 1 updated + 1 new row after Phase D's changes
- [ ] Run 4 demonstrated the delete blind spot — `ORD-001` remains in the target after deletion upstream
- [ ] Notebook run top-to-bottom with your own `YOUR_SCHEMA` value, no errors